# Vectorized Transformations, String Ops, Binning, np.select, Sessionization

## What you'll practice

Vectorization = express the operation on whole columns so it runs in C. `.str` for strings, `pd.cut` for bins, `np.where`/`np.select` for if/elif logic, `groupby().diff()/cumsum()`
for stateful sequences. You'll benchmark `apply` vs vectorized once — then never write row-wise `apply` for arithmetic again.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/pandas/test_09_03_transformations_vectorized.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/09_pandas/03_transformations_vectorized.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 140)
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
orders_raw = pd.read_csv(DATA / "csv" / "orders.csv", dtype=str, keep_default_na=False)
print(orders_raw.shape); orders_raw.head()

## Exercise 1 — 🟢 Easy · Arithmetic columns with assign

**Implement:** `add_amounts(df: pd.DataFrame, tax_rate: float = 0.18) -> pd.DataFrame`

Return a **new** frame with `amount = quantity * unit_price`, `tax = round(amount * tax_rate, 2)`, `total = amount + tax`. The input must not be modified.

**Input:** `df: pd.DataFrame, tax_rate: float = 0.18`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_add_amounts` in `tests/pandas/test_09_03_transformations_vectorized.py`.

**Check:** `pytest tests/pandas/test_09_03_transformations_vectorized.py -k test_add_amounts`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Vectorized string cleanup

**Implement:** `clean_text_columns(df: pd.DataFrame) -> pd.DataFrame`

On a copy: `channel` → stripped lowercase; `status` and `country` → stripped uppercase; `name` (if present) → whitespace-collapsed title case; empty strings in these columns → missing (`pd.NA`/NaN). Use `.str` methods only.

**Input:** `df: pd.DataFrame`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_clean_text_columns` in `tests/pandas/test_09_03_transformations_vectorized.py`.

**Check:** `pytest tests/pandas/test_09_03_transformations_vectorized.py -k test_clean_text_columns`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Binning with pd.cut

**Implement:** `add_buckets(df: pd.DataFrame) -> pd.DataFrame`

Add `amount_bucket` with `pd.cut` using bins `[0, 1000, 10000, inf)` (left-closed: `right=False`) and labels `"<1k", "1k-10k", "10k+"`, and `is_large = amount >= 10000` via a plain vectorized comparison. Negative/NaN amounts → NaN bucket.

**Input:** `df: pd.DataFrame`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_add_buckets` in `tests/pandas/test_09_03_transformations_vectorized.py`.

**Check:** `pytest tests/pandas/test_09_03_transformations_vectorized.py -k test_add_buckets`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · if/elif logic with np.select (and a benchmark)

**Implement:** `risk_level(df: pd.DataFrame) -> pd.Series`

Re-implement Level 1's `classify_risk` vectorized. Columns: `amount, currency, hour, country, home_country, failed_attempts` (NaN → 0). Rules in priority order:
HIGH if `amount > 200000` or `failed_attempts >= 3`; HIGH if foreign and `amount > 50000`; MEDIUM if `0 <= hour < 5` and `amount > 10000`; MEDIUM if currency != INR or foreign; else LOW.
Case-insensitive comparisons. Return a Series named `risk`. Then time it against `df.apply(row_function, axis=1)` on 100k rows.

**Input:** `df: pd.DataFrame`

**Expected output:** `pd.Series`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_risk_level` in `tests/pandas/test_09_03_transformations_vectorized.py`.

**Check:** `pytest tests/pandas/test_09_03_transformations_vectorized.py -k test_risk_level`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Vectorized sessionization · 🧩 Mini-project

**Implement:** `sessionize(events: pd.DataFrame, gap_minutes: int = 30) -> pd.DataFrame`

Given events with `user_id`, `ts` (datetime), add `session_id` = `f"{user_id}-{n}"` (n starting at 1 per user) where a new session starts when the gap from the user's previous event is
**> gap_minutes**. No Python loops over rows: sort, `groupby("user_id")["ts"].diff()`, compare, `cumsum` per user. Also return per-session stats as `result.attrs["sessions"]`: a DataFrame
`session_id, user_id, start, end, events, duration_min` sorted by session_id. Load `datasets/json/events.jsonl` (skip malformed lines yourself), parse `ts`, drop null user_ids, and sessionize.

**Input:** `events: pd.DataFrame, gap_minutes: int = 30`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_sessionize` in `tests/pandas/test_09_03_transformations_vectorized.py`.

**Check:** `pytest tests/pandas/test_09_03_transformations_vectorized.py -k test_sessionize`

**YOUR CODE ↓**